# Lab 2.3: DITTO — Few-Shot Stylistic Alignment

---

This lab covers **DITTO** (Demonstration ITerated Task Optimization), a method for aligning LLMs to specific stylistic preferences using only a handful of demonstrations (< 10 examples).

**Learning Objectives:**
- Understand the DITTO algorithm and its theoretical foundations
- Implement the two-stage DITTO pipeline (SFT → Online DPO)
- Use the actual DITTO codebase with DITTOTrainer
- Apply DITTO for personalized style transfer

**Reference**: Shaikh et al. (2025). *Aligning Language Models with Demonstrated Feedback*. ICLR 2025.

---

## 1. Motivation: Why DITTO?

### The Problem with Existing Approaches

<div>
<img src="diagrams/2.3.1.png" width="500"/>
</div>

**Key Insight**: Demonstrations implicitly define preferences—we can generate comparison data by treating user demos as preferred over model outputs.

---

## 2. How DITTO Works

DITTO operates in **two stages**:

<div>
<img src="diagrams/2.3.2.png" width="500"/>
</div>

---

## 3. Setup and Dependencies

In [1]:
!unzip -o ditto.zip

Archive:  ditto.zip
  inflating: ditto/run_ditto.py      
  inflating: ditto/__pycache__/ditto_trainer.cpython-312.pyc  
  inflating: ditto/__pycache__/dataset_utils.cpython-312.pyc  
  inflating: ditto/ditto-qwen2.5-7b-instruct.yaml  
  inflating: ditto/dataset_utils.py  
  inflating: ditto/ditto_trainer.py  
  inflating: ditto/sft_trainer.py    


In [2]:
# %%capture
# !pip install torch transformers accelerate peft trl datasets

In [3]:
import os
# Force single-GPU visibility for this notebook session (set before CUDA init).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys
import logging
import random
import warnings
import inspect
from typing import List, Dict, Any, Optional, Literal
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from transformers.trainer_callback import TrainerCallback

from peft import PeftModel, get_peft_model, LoraConfig, TaskType
from peft.utils import ModulesToSaveWrapper
from peft.tuners.tuners_utils import BaseTunerLayer

from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print(f"Using device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"USE_BF16={USE_BF16}")


Using device: cuda
CUDA_VISIBLE_DEVICES=0
USE_BF16=True


In [4]:
# Detect TRL version for API compatibility (same as run_ditto.py)
def _get_trainer_tokenizer_kwarg():
    """
    Newer TRL versions (>=0.12) use 'processing_class' instead of 'tokenizer'.
    Returns the correct keyword argument name.
    """
    sig = inspect.signature(DPOTrainer.__init__)
    if 'processing_class' in sig.parameters:
        return 'processing_class'
    elif 'tokenizer' in sig.parameters:
        return 'tokenizer'
    else:
        return 'processing_class'

TOKENIZER_KWARG = _get_trainer_tokenizer_kwarg()
print(f"TRL trainer tokenizer kwarg: {TOKENIZER_KWARG}")

TRL trainer tokenizer kwarg: processing_class


---

## 4. DITTO Helper Functions

These functions are from the DITTO codebase (`run_ditto.py`).

In [5]:
def is_openai_format(messages):
    """Check if messages are in OpenAI format (list of dicts with 'role' and 'content')."""
    if not isinstance(messages, list):
        return False
    if len(messages) == 0:
        return False
    return all(
        isinstance(m, dict) and "role" in m and "content" in m
        for m in messages
    )


def apply_chat_template(
    example,
    tokenizer,
    task: Literal["sft", "generation", "ditto", "dpo"]
):
    """
    Apply chat template to examples based on task type.
    From run_ditto.py - handles both SFT and DITTO/DPO formats.
    """
    if task in ["sft", "generation"]:
        messages = example["chosen"]
        example["text"] = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True if task == "generation" else False,
        )

    elif task in ["ditto", "dpo"]:
        if not is_openai_format(example["chosen"]):
            raise ValueError(
                f"Could not format example as dialogue for `{task}` task! Require OpenAI format"
            )

        if "prompt" in example and is_openai_format(example["prompt"]):
            prompt_messages = example["prompt"]
            chosen_messages = example["chosen"]
        else:
            prompt_messages = example["chosen"][:-1]
            chosen_messages = example["chosen"][-1:]

        example["text_prompt"] = tokenizer.apply_chat_template(prompt_messages, tokenize=False)
        example["text_chosen"] = tokenizer.apply_chat_template(chosen_messages, tokenize=False)

        if tokenizer.bos_token and example["text_chosen"] and example["text_chosen"].startswith(tokenizer.bos_token):
            example["text_chosen"] = example["text_chosen"][len(tokenizer.bos_token):]

        if task == "dpo" and "rejected" in example and example["rejected"]:
            if is_openai_format(example["rejected"]):
                if "prompt" in example and is_openai_format(example["prompt"]):
                    rejected_messages = example["rejected"]
                else:
                    rejected_messages = example["rejected"][-1:]

                example["text_rejected"] = tokenizer.apply_chat_template(rejected_messages, tokenize=False)

                if tokenizer.bos_token and example["text_rejected"] and example["text_rejected"].startswith(tokenizer.bos_token):
                    example["text_rejected"] = example["text_rejected"][len(tokenizer.bos_token):]
            else:
                example["text_rejected"] = example["rejected"]

    return example


def copy_adapter_weights(src_adapter_name, tgt_adapter_name, model):
    """
    Copy LoRA adapter weights from source adapter to target adapter.
    From run_ditto.py - used to initialize DITTO adapter from SFT.
    """
    lora_modules = [
        module for module in model.modules()
        if isinstance(module, (BaseTunerLayer, ModulesToSaveWrapper))
    ]

    with torch.no_grad():
        for model_module in lora_modules:
            if hasattr(model_module, 'lora_A') and src_adapter_name in model_module.lora_A.keys():
                model_module.lora_A[tgt_adapter_name].load_state_dict(
                    model_module.lora_A[src_adapter_name].state_dict()
                )
                model_module.lora_B[tgt_adapter_name].load_state_dict(
                    model_module.lora_B[src_adapter_name].state_dict()
                )

            if hasattr(model_module, 'lora_embedding_A') and src_adapter_name in model_module.lora_embedding_A.keys():
                model_module.lora_embedding_A[tgt_adapter_name].load_state_dict(
                    model_module.lora_embedding_A[src_adapter_name].state_dict()
                )
                model_module.lora_embedding_B[tgt_adapter_name].load_state_dict(
                    model_module.lora_embedding_B[src_adapter_name].state_dict()
                )

    logger.info(f"Copied adapter weights: {src_adapter_name} → {tgt_adapter_name}")


def save_adapter(model, tokenizer, output_dir, adapter_name):
    """Save a specific adapter to disk."""
    os.makedirs(output_dir, exist_ok=True)
    model.set_adapter(adapter_name)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    if hasattr(model, 'config'):
        model.config.save_pretrained(output_dir)
    logger.info(f"Adapter '{adapter_name}' saved to {output_dir}")

In [6]:
class EarlyStoppingCallback(TrainerCallback):
    """
    Stop training when loss falls below threshold.
    From run_ditto.py - used in SFT stage to prevent overfitting on few demonstrations.
    """

    def __init__(self, threshold=1.0):
        self.threshold = threshold

    def on_step_begin(self, args, state, control, **kwargs):
        if len(state.log_history) > 0:
            last_loss = None
            for k in state.log_history[::-1]:
                if "loss" in k:
                    last_loss = k["loss"]
                    break
            if last_loss is not None and last_loss < self.threshold:
                logger.info(f"Early stopping: loss {last_loss:.4f} < threshold {self.threshold}")
                control.should_training_stop = True

---

## 5. Understanding DITTO: Simplified Implementation

Before using the actual DITTOTrainer, let's understand the core logic with a simplified implementation.

In [7]:
class SimpleDITTOLogic:
    """
    SIMPLIFIED DITTO implementation for EDUCATIONAL PURPOSES ONLY.

    This class demonstrates the core DITTO algorithm:
    1. Generate negative samples from current policy
    2. Create comparisons using 70/20/10 split (expert/replay/intermodel)
    3. Update using DPO-style loss

    For actual training, use DITTOTrainer from ditto/ditto_trainer.py!
    """

    def __init__(self, model, tokenizer, train_dataset,
                 samples_per_prompt=3, frac_expert=0.7, frac_replay=0.2, frac_noisy=0.1):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.samples_per_prompt = samples_per_prompt

        # DITTO's comparison split
        self.frac_expert = frac_expert   # Expert demos > current model outputs
        self.frac_replay = frac_replay   # Expert demos > past model outputs
        self.frac_noisy = frac_noisy     # Current > past (intermodel)

        # Cache stores generated samples across iterations
        self.sample_cache = {}  # {iteration: {prompt: [samples]}}

    def sample_negatives(self, iteration):
        """
        Generate negative samples from the current policy.
        These will be compared against expert demonstrations.
        """
        print(f"  Generating {self.samples_per_prompt} samples per prompt...")
        self.model.eval()
        self.sample_cache[iteration] = {}

        for example in self.train_dataset:
            prompt = example["prompt"]
            samples = []

            for _ in range(self.samples_per_prompt):
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=256,
                        do_sample=True,
                        temperature=1.0,
                        pad_token_id=self.tokenizer.pad_token_id,
                    )
                response = self.tokenizer.decode(
                    outputs[0][inputs.input_ids.shape[1]:],
                    skip_special_tokens=True
                )
                samples.append(response)

            self.sample_cache[iteration][prompt] = samples

        self.model.train()
        return self.sample_cache[iteration]

    def build_comparisons(self, iteration):
        """
        Build comparison pairs using DITTO's 70/20/10 split:

        - 70% Expert:     demo ≻ current_sample   (expert is better than current)
        - 20% Replay:     demo ≻ past_sample      (expert is better than past)
        - 10% Intermodel: current ≻ past          (current is better than past)

        This mix provides a smooth learning signal that:
        - Anchors to expert quality (70%)
        - Prevents forgetting (20% replay)
        - Encourages continued improvement (10% intermodel)
        """
        print(f"  Building comparisons with {self.frac_expert:.0%}/{self.frac_replay:.0%}/{self.frac_noisy:.0%} split...")
        comparisons = []

        for example in self.train_dataset:
            prompt = example["prompt"]
            expert_response = example["chosen"]

            current_samples = self.sample_cache.get(iteration, {}).get(prompt, [])

            for rejected in current_samples:
                r = random.random()

                if r < self.frac_expert:  # Expert comparison (70%)
                    comparisons.append({
                        "prompt": prompt,
                        "chosen": expert_response,
                        "rejected": rejected,
                        "type": "expert"
                    })
                elif r < self.frac_expert + self.frac_replay and iteration > 0:  # Replay (20%)
                    past_iter = random.randint(0, iteration - 1)
                    if past_iter in self.sample_cache and prompt in self.sample_cache[past_iter]:
                        past_sample = random.choice(self.sample_cache[past_iter][prompt])
                        comparisons.append({
                            "prompt": prompt,
                            "chosen": expert_response,
                            "rejected": past_sample,
                            "type": "replay"
                        })
                elif iteration > 0:  # Intermodel (10%)
                    past_iter = random.randint(0, iteration - 1)
                    if past_iter in self.sample_cache and prompt in self.sample_cache[past_iter]:
                        past_sample = random.choice(self.sample_cache[past_iter][prompt])
                        comparisons.append({
                            "prompt": prompt,
                            "chosen": rejected,  # Current is "better" than past
                            "rejected": past_sample,
                            "type": "intermodel"
                        })

        # Count types
        type_counts = {}
        for c in comparisons:
            t = c.get("type", "unknown")
            type_counts[t] = type_counts.get(t, 0) + 1
        print(f"  Comparison types: {type_counts}")

        return comparisons

print("SimpleDITTOLogic class")

SimpleDITTOLogic class


---

## 6. Create Style Demonstration Data

For DITTO, we need **high-quality demonstrations** that exemplify the target style.

In [8]:
# Style: "The Friendly Expert" - conversational, uses analogies, ends with signature
DEMONSTRATIONS = [
    {
        "prompt": [{"role": "user", "content": "What is machine learning?"}],
        "chosen": [{"role": "assistant", "content": """Ah, great question! So here's the thing about machine learning—it's basically teaching computers to learn from experience, just like how you learned to ride a bike.

Instead of programming every rule explicitly, we show the computer lots of examples and let it figure out the patterns. Think of it like this: instead of telling a computer "a cat has pointy ears, whiskers, and says meow," you show it thousands of cat pictures until it goes "Ohhh, I see what makes a cat a cat!"

~Cheers, and happy learning! 🚀"""}]
    },
    {
        "prompt": [{"role": "user", "content": "How does the internet work?"}],
        "chosen": [{"role": "assistant", "content": """Oh, the internet! It's like a giant game of telephone, but way more reliable.

When you visit a website, your computer breaks your request into tiny packets—think of them like postcards. These postcards travel through a network of routers (like post offices), each one reading the address and sending it closer to the destination.

The wild part? These packets might take completely different routes but still arrive in the right order!

~Cheers, and happy learning! 🚀"""}]
    },
    {
        "prompt": [{"role": "user", "content": "Why is exercise good for you?"}],
        "chosen": [{"role": "assistant", "content": """So here's the thing about exercise—your body is basically a "use it or lose it" machine!

When you exercise, you're sending signals to your body saying "hey, we need to be stronger/faster/more enduring." Your heart pumps more efficiently, your muscles grow, and your brain releases feel-good chemicals called endorphins.

It's like upgrading your hardware AND software at the same time. Pretty cool deal, right?

~Cheers, and happy learning! 🚀"""}]
    },
    {
        "prompt": [{"role": "user", "content": "What causes climate change?"}],
        "chosen": [{"role": "assistant", "content": """Ah, this is an important one! Climate change is basically Earth running a fever.

Here's what's happening: we're burning fossil fuels (coal, oil, gas) which release CO2 into the atmosphere. This CO2 acts like a blanket, trapping heat—similar to how your car gets hot when parked in the sun with windows up.

The result? Oceans warming, ice melting, weather getting more extreme. The science is solid: we need to reduce emissions!

~Cheers, and happy learning! 🚀"""}]
    },
    {
        "prompt": [{"role": "user", "content": "How do vaccines work?"}],
        "chosen": [{"role": "assistant", "content": """Oh, vaccines are SO clever! Think of your immune system as a security team.

A vaccine is basically a wanted poster—it shows your immune system what the bad guy (virus/bacteria) looks like, WITHOUT actually making you sick. Your body then creates specialized defenders (antibodies) that remember this threat.

So if the real pathogen ever shows up? Your immune system goes "I KNOW YOU!" and neutralizes it fast.

~Cheers, and happy learning! 🚀"""}]
    }
]

print(f"Created {len(DEMONSTRATIONS)} demonstrations with 'Friendly Expert' style")

Created 5 demonstrations with 'Friendly Expert' style


---

## 7. Prepare Data for SFT and DITTO

In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded tokenizer: {MODEL_NAME}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request:

Loaded tokenizer: Qwen/Qwen2.5-1.5B-Instruct


In [10]:
def prepare_sft_dataset(demonstrations, tokenizer):
    """
    Prepare dataset for SFT stage.
    Uses apply_chat_template to create 'text' field.
    """
    sft_data = []
    for demo in demonstrations:
        example = {"chosen": demo["prompt"] + demo["chosen"]}
        example = apply_chat_template(example, tokenizer, task="sft")
        sft_data.append({"text": example["text"]})
    return Dataset.from_list(sft_data)


def prepare_ditto_dataset(demonstrations, tokenizer):
    """
    Prepare dataset for DITTO stage.
    Creates 'prompt' and 'chosen' fields in the format expected by DITTOTrainer.
    """
    ditto_data = []
    for demo in demonstrations:
        # Format prompt with generation prompt
        prompt_text = tokenizer.apply_chat_template(
            demo["prompt"],
            tokenize=False,
            add_generation_prompt=True
        )
        # Chosen is the assistant response
        chosen_text = demo["chosen"][0]["content"]

        ditto_data.append({
            "prompt": prompt_text,
            "chosen": chosen_text,
            # Note: 'rejected' will be generated online by DITTO
        })
    return Dataset.from_list(ditto_data)


# Create datasets
sft_dataset = prepare_sft_dataset(DEMONSTRATIONS, tokenizer)
ditto_dataset = prepare_ditto_dataset(DEMONSTRATIONS, tokenizer)

print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"DITTO dataset: {len(ditto_dataset)} examples")
print(f"\nSFT example (first 300 chars):\n{sft_dataset[0]['text'][:300]}")

SFT dataset: 5 examples
DITTO dataset: 5 examples

SFT example (first 300 chars):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
Ah, great question! So here's the thing about machine learning—it's basically teaching computers to learn from experience, just


---

## 8. Load Model and Configure LoRA

In [11]:
# Load base model (single-GPU, trainer-friendly loading)
model_dtype = torch.bfloat16 if USE_BF16 else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
    trust_remote_code=True,
)
model.to(DEVICE)
model.config.use_cache = False

print(f"Loaded model: {MODEL_NAME}")
print(f"Model dtype: {model_dtype}")
print(f"Model device: {next(model.parameters()).device}")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
`torch_dtype` is deprecated! Use `dtype` instead!
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


Loaded model: Qwen/Qwen2.5-1.5B-Instruct
Model dtype: torch.bfloat16
Model device: cuda:0


In [12]:
# LoRA configuration (same as run_ditto.py)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM
)

print(f"LoRA config: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

LoRA config: r=16, alpha=32
Target modules: {'up_proj', 'k_proj', 'down_proj', 'q_proj', 'gate_proj', 'o_proj', 'v_proj'}


---

## 9. STAGE 1: SFT on Demonstrations

The first stage teaches the model the basic style through imitation.

This follows the exact pattern from `run_ditto.py`:

In [13]:
# =========================================================================
# STAGE 1: SFT Training (exactly as in run_ditto.py)
# =========================================================================
logger.info("=" * 60)
logger.info("STAGE 1/2: SFT Training")
logger.info("=" * 60)

# Add SFT adapter
model = get_peft_model(model, lora_config, adapter_name="sft")
model.set_adapter("sft")
model.print_trainable_parameters()

INFO:__main__:============================================================
INFO:__main__:STAGE 1/2: SFT Training
INFO:__main__:============================================================


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [14]:
# SFT output directory
sft_output_dir = "./outputs/ditto_sft"

# Basic dataset sanity: drop empty/blank rows to avoid degenerate batches
before_n = len(sft_dataset)
sft_dataset = sft_dataset.filter(lambda ex: isinstance(ex["text"], str) and len(ex["text"].strip()) > 0)
after_n = len(sft_dataset)
print(f"SFT dataset size: {before_n} -> {after_n} after empty-text filtering")

sft_config = SFTConfig(
    output_dir=sft_output_dir,
    max_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-6,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    weight_decay=0.0,
    bf16=USE_BF16,
    fp16=False,
    gradient_checkpointing=False,
    logging_steps=1,
    save_strategy="no",
    logging_nan_inf_filter=True,
    dataset_text_field="text",
    optim="adamw_torch",
    report_to="none",
)

sft_trainer_kwargs = {
    "model": model,
    "args": sft_config,
    "train_dataset": sft_dataset,
    TOKENIZER_KWARG: tokenizer,
}
sft_trainer = SFTTrainer(**sft_trainer_kwargs)


Filter:   0%|          | 0/5 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFT dataset size: 5 -> 5 after empty-text filtering


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [15]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.bos_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False


In [16]:
b = next(iter(sft_trainer.get_train_dataloader()))
b = {k:v.to(model.device) for k,v in b.items() if hasattr(v, "to")}
with torch.no_grad():
    print(model(**b).loss)


tensor(3.1706, device='cuda:0')


In [17]:
assert False

AssertionError: 

In [18]:
import torch, math

dl = sft_trainer.get_train_dataloader()
model.eval()

for i, b in enumerate(dl):
    b = {k: v.to(model.device) for k, v in b.items() if hasattr(v, "to")}
    valid = (b["labels"] != -100).sum().item()
    max_id = b["input_ids"].max().item()
    min_id = b["input_ids"].min().item()
    with torch.no_grad():
        loss = model(**b).loss
    print(i, "valid_labels=", valid, "id_range=", (min_id, max_id), "loss=", float(loss))
    if not torch.isfinite(loss):
        print("BAD BATCH AT", i)
        break



0 valid_labels= 138 id_range= (0, 151645) loss= 3.1705994606018066
1 valid_labels= 136 id_range= (0, 151645) loss= 3.5234906673431396
2 valid_labels= 143 id_range= (0, 151645) loss= 3.15073561668396
3 valid_labels= 152 id_range= (0, 151645) loss= 2.8358545303344727
4 valid_labels= 135 id_range= (0, 151645) loss= 3.1771059036254883


In [19]:
print("hf_device_map:", getattr(model, "hf_device_map", None))
print("first param device:", next(model.parameters()).device)

import os, torch
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.device_count():", torch.cuda.device_count())


hf_device_map: None
first param device: cuda:0
CUDA_VISIBLE_DEVICES: 0
torch.cuda.device_count(): 1


In [20]:
logger.info("Starting SFT training...")
sft_trainer.train()

# Save SFT adapter
save_adapter(model, tokenizer, sft_output_dir, "sft")
logger.info(f"SFT adapter saved to {sft_output_dir}")

INFO:__main__:Starting SFT training...


Step,Training Loss
1,3.147624
2,3.497583
3,3.136848
4,2.819720
5,3.153730
6,3.136258
7,3.148021


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:__main__:Adapter 'sft' saved to ./outputs/ditto_sft
INFO:__main__:SFT adapter saved to ./outputs/ditto_sft


---

## 10. STAGE 2: DITTO Training with Actual DITTOTrainer

Now we use the **actual DITTOTrainer** from `ditto/ditto_trainer.py`, exactly as it's used in `run_ditto.py`.

The DITTOTrainer handles:
- Online generation of negative samples
- Building comparisons with the 70/20/10 split
- DPO-style optimization with adapter switching

In [21]:
# Import the actual DITTOTrainer
# Add the ditto directory to path
sys.path.insert(0, 'ditto')

from ditto_trainer import DITTOTrainer

print("Imported DITTOTrainer from ditto/ditto_trainer.py")

Imported DITTOTrainer from ditto/ditto_trainer.py


In [22]:
# =========================================================================
# STAGE 2: DITTO Training (exactly as in run_ditto.py)
# =========================================================================
logger.info("=" * 60)
logger.info("STAGE 2/2: DITTO Training (on top of SFT)")
logger.info("=" * 60)

# DITTO hyperparameters
ditto_output_dir = "./outputs/ditto_final"
DITTO_LEARNING_RATE = 5e-6
DITTO_MAX_STEPS = 30
DITTO_WARMUP_RATIO = 0.1
DITTO_BATCH_SIZE = 8
BETA = 0.1
MAX_LENGTH = 128
MAX_PROMPT_LENGTH = 512

# Add DITTO adapter and copy weights from SFT
model.add_adapter("ditto", lora_config)
model.set_adapter("ditto")
copy_adapter_weights("sft", "ditto", model)

print("DITTO adapter created and initialized from SFT weights")

INFO:__main__:============================================================
INFO:__main__:STAGE 2/2: DITTO Training (on top of SFT)
INFO:__main__:============================================================
INFO:__main__:Copied adapter weights: sft → ditto


DITTO adapter created and initialized from SFT weights


In [23]:
# Create training arguments for DITTO
@dataclass
class DITTOArgs(TrainingArguments):
    """Extended training arguments for DITTO."""
    frac_expert: float = 0.7
    frac_replay: float = 0.2
    frac_noisy: float = 0.1
    rescale_batch: int = 3
    resample_rate: int = 10
    bootstrap_count: int = 10
    reset_rate: int = -1

ditto_training_args = DITTOArgs(
    output_dir=ditto_output_dir,
    learning_rate=DITTO_LEARNING_RATE,
    max_steps=DITTO_MAX_STEPS,
    per_device_train_batch_size=DITTO_BATCH_SIZE,
    gradient_accumulation_steps=2,
    warmup_ratio=DITTO_WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=500,
    bf16=True,
    remove_unused_columns=False,
    # DITTO specific
    frac_expert=0.7,
    frac_replay=0.2,
    frac_noisy=0.1,
    rescale_batch=3,
    resample_rate=100,
    bootstrap_count=1,  # Reduced for demo
)

print(f"DITTO training args configured:")
print(f"  - Learning rate: {DITTO_LEARNING_RATE}")
print(f"  - Max steps: {DITTO_MAX_STEPS}")
print(f"  - Beta: {BETA}")
print(f"  - Comparison split: {ditto_training_args.frac_expert}/{ditto_training_args.frac_replay}/{ditto_training_args.frac_noisy}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


DITTO training args configured:
  - Learning rate: 5e-06
  - Max steps: 30
  - Beta: 0.1
  - Comparison split: 0.7/0.2/0.1


In [25]:
import importlib
import dataset_utils
import ditto_trainer

# RELOAD modules to pick up your changes to dataset_utils.py
importlib.reload(dataset_utils)
importlib.reload(ditto_trainer)
from ditto_trainer import DITTOTrainer

# Initialize DITTOTrainer (exactly as in run_ditto.py)
ditto_trainer = DITTOTrainer(
    model=model,
    ref_adapter_name="sft",        # Reference model is the SFT adapter
    model_adapter_name="ditto",    # Training adapter
    args=ditto_training_args,
    beta=BETA,
    train_dataset=ditto_dataset,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    loss_type="sigmoid",
)

print("DITTOTrainer initialized")
print(f"  - Reference adapter: 'sft'")
print(f"  - Training adapter: 'ditto'")

DITTOTrainer initialized
  - Reference adapter: 'sft'
  - Training adapter: 'ditto'


In [26]:
# Run DITTO training
logger.info("Starting DITTO training...")
train_result = ditto_trainer.train()

# Log metrics
metrics = train_result.metrics
metrics["train_samples"] = len(ditto_dataset)
ditto_trainer.log_metrics("train", metrics)
ditto_trainer.save_metrics("train", metrics)
ditto_trainer.save_state()

logger.info("*** DITTO Training complete ***")

INFO:__main__:Starting DITTO training...
Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


STARTING EPOCH: 0


  0%|          | 0/5 [00:00<?, ?it/s]Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'num_return_sequences', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 20%|██        | 1/5 [00:09<00:39,  9.80s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 60%|██████    | 3/5 [00:17<00:10,  5.23s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|█████████

Step,Training Loss
5,0.692456
10,0.693536
15,0.691729
20,0.691249
25,0.691835
30,0.690434


...

STARTING EPOCH: 11
STARTING EPOCH: 12
STARTING EPOCH: 13
STARTING EPOCH: 14
STARTING EPOCH: 15
STARTING EPOCH: 16
STARTING EPOCH: 17
STARTING EPOCH: 18
STARTING EPOCH: 19
STARTING EPOCH: 20


  0%|          | 0/5 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 20%|██        | 1/5 [00:08<00:34,  8.51s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 60%|██████    | 3/5 [00:16<00:10,  5.24s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for mo

STARTING EPOCH: 21
STARTING EPOCH: 22
STARTING EPOCH: 23
STARTING EPOCH: 24
STARTING EPOCH: 25
STARTING EPOCH: 26
STARTING EPOCH: 27
STARTING EPOCH: 28
STARTING EPOCH: 29


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qw

***** train metrics *****
  epoch                    =       30.0
  total_flos               =        0GF
  train_loss               =     0.6919
  train_runtime            = 0:01:36.19
  train_samples            =          5
  train_samples_per_second =       4.99
  train_steps_per_second   =      0.312


In [27]:
# Save DITTO adapter (from run_ditto.py)
#model.delete_adapter("sft")  # Clean up
save_adapter(model, tokenizer, ditto_output_dir, "ditto")
logger.info(f"DITTO adapter saved to {ditto_output_dir}")

# Save config
if ditto_trainer.accelerator.is_main_process:
    ditto_trainer.model.config.use_cache = True
    ditto_trainer.model.config.save_pretrained(ditto_output_dir)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qw

In [28]:
# Summary
logger.info("=" * 60)
logger.info("Training Complete! Adapters saved:")
logger.info(f"  1. SFT:   {sft_output_dir}")
logger.info(f"  2. DITTO: {ditto_output_dir}")
logger.info("=" * 60)

INFO:__main__:============================================================
INFO:__main__:Training Complete! Adapters saved:
INFO:__main__:  1. SFT:   ./outputs/ditto_sft
INFO:__main__:  2. DITTO: ./outputs/ditto_final
INFO:__main__:============================================================


---

## 11. Evaluation: Compare SFT vs DITTO

In [29]:
def generate_response(model, tokenizer, prompt, max_new_tokens=200, temperature=0.7):
    """Generate a response from the model."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [30]:
# Reload model with both adapters for comparison
from peft import PeftModel

# Load base model fresh
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)

# Load SFT adapter
model_with_adapters = PeftModel.from_pretrained(base_model, f"{sft_output_dir}/sft", adapter_name="sft")

# Load DITTO adapter
model_with_adapters.load_adapter(f"{ditto_output_dir}/ditto", adapter_name="ditto")

print("Loaded both SFT and DITTO adapters for comparison")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json "HTTP/1.1 200 OK"


Loaded both SFT and DITTO adapters for comparison


In [31]:
# Test prompts (different from training!)
test_prompts = [
    "What is photosynthesis?",
    "How does encryption protect data?",
    "Why do we need sleep?",
]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")

    # SFT response
    model_with_adapters.set_adapter("sft")
    print("\n--- SFT Model ---")
    print(generate_response(model_with_adapters, tokenizer, prompt))

    # DITTO response
    model_with_adapters.set_adapter("ditto")
    print("\n--- DITTO Model ---")
    print(generate_response(model_with_adapters, tokenizer, prompt))


PROMPT: What is photosynthesis?

--- SFT Model ---


Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize food from carbon dioxide and water using chlorophyll as an enzyme. The overall chemical equation for photosynthesis is:

6 CO2 + 6 H2O + light energy → C6H12O6 (glucose) + 6 O2

This process occurs in the leaves of plants, where chloroplasts contain chlorophyll that captures light energy. Chlorophyll absorbs light energy from the sun, and uses it to convert carbon dioxide and water into glucose (a type of sugar) and oxygen.

Glucose serves as an energy source for the plant, while oxygen is released into the atmosphere as a waste product. This process is crucial for life on Earth, as it provides oxygen for most living organisms and produces organic compounds like glucose that serve as building blocks for cellular respiration.

--- DITTO Model ---
Photosynthesis is the process by which green plants and some other organisms use sunlight to transform carbon dioxide and water into oxygen 

---

## 12. DITTO Best Practices

### When to Use DITTO
- Personalizing LLM to a specific writing voice/brand
- Domain adaptation with very few examples
- Style transfer without paired preference data
- Rapid prototyping of customized assistants

### Hyperparameters (from run_ditto.py)

| Stage | Parameter | Recommended Value |
|-------|-----------|------------------|
| SFT | Learning Rate | 3e-5 |
| SFT | Early Stop Loss | < 1.0 - 1.25 |
| DITTO | Learning Rate | 5e-6 (very low) |
| DITTO | Beta (β) | 0.1 |
| DITTO | Max Steps | 30 |
| DITTO | Bootstrap Count | 10 |
| DITTO | Resample Rate | 10 |
| DITTO | Comparison Split | 70% Expert, 20% Replay, 10% Intermodel |

### Critical: Reference Model

> **Keep π_ref FIXED!** The SFT adapter serves as the reference model and must NOT be updated during DITTO training. This is handled automatically by DITTOTrainer through `ref_adapter_name`.

---

## Key Takeaways

1. **DITTO = SFT + Online DPO**: First learn basic style, then refine with self-generated comparisons.

2. **Few-shot alignment works**: DITTO achieves strong personalization with < 10 demonstrations.

3. **DITTOTrainer handles complexity**: The actual implementation manages online sampling, comparison building, and adapter switching.

4. **Three comparison types**: Expert (70%) + Replay (20%) + Intermodel (10%) provides smooth learning signal.

5. **Quality > quantity for demos**: 5 excellent, consistent examples beat 50 inconsistent ones.

---

## References

- Shaikh et al. (2025). Aligning Language Models with Demonstrated Feedback. *ICLR 2025*.
- Rafailov et al. (2023). Direct Preference Optimization. *NeurIPS*.
- Souza, T. (2024). Taming LLMs. *GitHub*.

**Next**: Lab 2.4 — Exercise: Build Your Own Style Dataset